# Beyond deep learning: the same algebra as shortest paths

The min-plus semiring is the native algebra of **dynamic programming**. Tropical matmul
`C_ij = min_k (A_ik + B_kj)` is exactly "best one-hop-extended path" — so the *same*
operation that is a tropical NN layer is the Floyd–Warshall update. `tikitorch.algorithmic`
ships the DP algorithms; `tikitorch.classical` ships order-based classical-ML methods.


In [ ]:
# --- Setup: install TikiTorch (run once) ---
# Until a tagged package is published, Colab can install TikiTorch from `main`.
try:
    import tikitorch  # already installed?
except ImportError:
    REPO_URL = "https://github.com/nyuolab/tikitorch.git"
    import subprocess, sys, os
    if not os.path.exists("tikitorch_repo"):
        subprocess.run(
            ["git", "clone", "--branch", "main", "--single-branch",
             REPO_URL, "tikitorch_repo"],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", "tikitorch_repo"],
        check=True,
    )
    import tikitorch
import torch
print("TikiTorch", getattr(tikitorch, "__version__", "?"), "| torch", torch.__version__)


## Shortest paths = repeated min-plus matmul

`floyd_warshall` (all-pairs) is the min-plus matrix closure; `bellman_ford` is
single-source relaxation. Inputs are batched edge-weight matrices `(B, N, N)`, with
`inf` for an absent edge.


In [ ]:
import torch
from tikitorch.algorithmic import floyd_warshall, bellman_ford
INF = float("inf")
W = torch.tensor([[[0., 2, INF, 1],
                   [INF, 0, 3, INF],
                   [INF, INF, 0, 1],
                   [INF, 4, INF, 0]]])              # one graph, 4 nodes
print("all-pairs shortest paths:\n", floyd_warshall(W)[0])
print("single-source (from node 0):", bellman_ford(W, src=0)[0].tolist())


## More DP, same primitive

Longest-increasing-subsequence (max-plus), min-coin change, and 0/1 knapsack are all
tropical DP. The `*_selection` variants walk the DP table back through its argmin/argmax
to recover the actual solution — the same argmin routing that carries gradients in a
tropical layer.


In [ ]:
from tikitorch.algorithmic import lis_lengths, min_coin_selection, knapsack_selection
print("LIS DP lengths:", lis_lengths(torch.tensor([[3, 1, 4, 1, 5, 9, 2, 6]]))[0].tolist())
print("min coins for 6 from {1,3,4}:",
      min_coin_selection(torch.tensor([[1, 3, 4]]), torch.tensor([6]))[0].tolist())
print("knapsack pick (cap 4, w={1,3,4}, v={15,20,30}):",
      knapsack_selection(torch.tensor([[1, 3, 4]]), torch.tensor([[15, 20, 30]]),
                         torch.tensor([4]))[0].tolist())


Because the layer and the algorithm share a primitive, a tropical net has the right
inductive bias to **learn** these algorithms from examples — see
`tikitorch.algorithmic.build_reasoner` / `evaluate_task` and the README's algorithmic
reasoning benchmarks.


## Classical tropical machine learning

`tikitorch.classical` gives order-based analogues built on the tropical metric
`tropical_distance`: clustering, nearest-centroid, a max-margin SVM, and PCA.


In [ ]:
import torch
from tikitorch.classical import (tropical_kmeans, TropicalNearestCentroid,
                                  TropicalSVM, tropical_pca)
torch.manual_seed(0)
# The tropical metric is PROJECTIVE (it quotients out the all-ones direction), so the
# classes must differ in a *direction*, not by a global shift the metric would ignore.
mA = torch.tensor([4., 0, 0, 0]); mB = torch.tensor([0., 0, 4, 0])
X = torch.cat([torch.randn(40, 4) * 0.5 + mA, torch.randn(40, 4) * 0.5 + mB])
y = torch.cat([torch.zeros(40), torch.ones(40)]).long()

labels, centroids = tropical_kmeans(X, k=2)
print("k-means cluster sizes:", torch.bincount(labels.long()).tolist())

nc = TropicalNearestCentroid().fit(X, y)
print("nearest-centroid train accuracy:", round(float(nc.score(X, y)), 3))

svm = TropicalSVM().fit(X, y)
print("tropical-SVM train accuracy:    ", round(float(svm.score(X, y)), 3))

Z = tropical_pca(X, k=2)
print("tropical PCA embedding shape:   ", tuple(Z.shape))


## The through-line

A tropical NN layer, a shortest-path relaxation, a DP table, and a max-plus schedule are
**one operation** — `min`/`max` of sums — wearing different hats. That is why this
substrate can suit both constrained hardware recipes *and* combinatorial reasoning.

Concept companion: **Guide → Beyond deep learning**.
